# 01. ResNet18 학습 + Grad-CAM 추출 (Kermany 흉부 X-ray)

**프로젝트:** AI는 폐렴을 어떻게 보나 — 그리고 어떻게 속았나 (`xray-gradcam-story`)

이 노트북이 하는 일
1. 구글 드라이브의 `ChestXRay2017.zip`을 Colab 디스크에 푼다
2. **환자 단위**로 train / val을 나눈다 (test 폴더는 최종 시험용으로 그대로 둔다)
3. 같은 ResNet18을 두 가지 조건으로 학습한다

| 모델 | 학습·검증 사진 | 시험 사진 | 의미 |
|---|---|---|---|
| **clean** | 늘려서 224×224 (여백 없음) | 여백 없음 | 기준 모델 |
| **padded** | 비율 유지 + 검은 여백 (letterbox) | 여백 있음 / 없음 둘 다 | DACON 때의 학습 조건 재현 |

4. Grad-CAM 7×7 지도와 **테두리 집중 비율**을 계산한다
5. 웹 페이지에 넣을 `story.json`을 드라이브 `outputs/`에 저장한다

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택

**데이터 출처:** Kermany, D., Zhang, K., Goldbaum, M. (2018). *Labeled Optical Coherence Tomography (OCT) and Chest X-Ray Images for Classification.* Mendeley Data, V2. doi:10.17632/rscbjbr9sj.2 (CC BY 4.0)

> 이 노트북의 숫자는 **Kermany 재실험 결과**이며, DACON 과제의 수치(0.9407, 0.68/0.21 등)와는 데이터 구성이 달라 직접 비교하지 않는다.

In [ ]:
# ===== 1. Mount Google Drive =====


# google.colab.drive: 구글 드라이브를 Colab의 폴더처럼 연결해 주는 도구
from google.colab import drive

# mount('/content/drive'): 내 드라이브가 /content/drive/MyDrive 아래에 보이게 된다
drive.mount('/content/drive')


# → 구글 드라이브를 연결해서, 업로드해 둔 ChestXRay2017.zip을 읽을 수 있게 하겠다
#   (처음 실행하면 구글 계정 허용 창이 뜬다 → 허용)

In [ ]:
# ===== 2. Paths & unzip =====


# os: 폴더·파일 경로를 다루는 기본 모듈
import os
# glob: '*' 같은 패턴으로 파일 목록을 찾는 모듈
import glob

# DRIVE_DIR: 드라이브에 만든 프로젝트 폴더
DRIVE_DIR = '/content/drive/MyDrive/xray-gradcam-story'
# ZIP_PATH: 업로드한 데이터 zip 위치
ZIP_PATH = f'{DRIVE_DIR}/data/ChestXRay2017.zip'
# WORK_DIR: 압축을 풀 곳 — 드라이브가 아니라 Colab 자체 디스크라서 읽기가 훨씬 빠르다
WORK_DIR = '/content/data'
# OUT_DIR: 결과(모델 가중치, 표, JSON)를 저장할 드라이브 폴더 — 런타임이 꺼져도 남는다
OUT_DIR = f'{DRIVE_DIR}/outputs'
# exist_ok=True: 폴더가 이미 있어도 오류 없이 넘어간다
os.makedirs(OUT_DIR, exist_ok=True)

# 아직 풀지 않았을 때만 압축 해제 (셀을 다시 실행해도 중복으로 풀지 않게)
if not os.path.exists(WORK_DIR):
    # !: 리눅스 명령 실행, -q: 진행 메시지 숨김, -d: 풀 위치
    !unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"

# recursive=True: 하위 폴더까지 뒤져 'train' 폴더를 찾는다 (맥에서 만든 zip의 __MACOSX 폴더는 제외)
TRAIN_DIR = [d for d in glob.glob(f'{WORK_DIR}/**/train', recursive=True) if '__MACOSX' not in d][0]
# test 폴더는 train 폴더와 같은 위치에 있다
TEST_DIR = os.path.join(os.path.dirname(TRAIN_DIR), 'test')
print('train:', TRAIN_DIR)
print('test :', TEST_DIR)


# → zip을 Colab 디스크에 풀고, train / test 폴더 위치를 찾겠다

In [ ]:
# ===== 3. Imports & reproducibility =====


# re: 정규표현식(파일 이름에서 환자 번호 뽑기), io·base64·json: 결과 파일 만들기, random·time: 난수·시간
import re, io, json, base64, random, time
# numpy: 숫자 배열 계산, pandas: 표(DataFrame) 다루기
import numpy as np
import pandas as pd
# PIL.Image: 사진 열기·크기 바꾸기
from PIL import Image
# tqdm: 반복문 진행 막대
from tqdm.auto import tqdm
# torch: 딥러닝 라이브러리 PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
# Dataset·DataLoader: 사진을 묶음(batch)으로 모델에 공급하는 도구
from torch.utils.data import Dataset, DataLoader
# torchvision.models: ImageNet으로 미리 학습된 모델 모음
from torchvision import models
# roc_auc_score: AUC 계산, confusion_matrix: 혼동행렬(FN/FP 개수)
from sklearn.metrics import roc_auc_score, confusion_matrix
# StratifiedGroupKFold: 클래스 비율은 유지하면서 같은 환자는 한쪽에만 들어가게 나누는 도구
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
# 파이썬·넘파이·파이토치의 난수 시작점을 고정 → 다시 실행해도 비슷한 결과
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# deterministic: GPU 연산도 매번 같은 방식으로, benchmark=False: 속도 최적화 탐색 끔(결과 흔들림 방지)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU가 있으면 'cuda', 없으면 'cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# USE_AMP: GPU일 때만 16비트 혼합 정밀도(빠른 계산)를 쓴다
USE_AMP = DEVICE == 'cuda'
print('device:', DEVICE)


# → 필요한 도구를 불러오고, 난수를 고정한 뒤 GPU 사용 여부를 확인하겠다
#   (device: cuda 가 안 나오면 런타임 → 런타임 유형 변경 → T4 GPU 후 처음부터 다시)

In [ ]:
# ===== 4. Config =====


CFG = dict(
    # img_size: ResNet18 입력 크기 (224×224)
    img_size=224,
    # batch_size: 한 번에 GPU로 보내는 사진 수
    batch_size=32,
    # epochs: 학습 데이터를 몇 바퀴 볼지
    epochs=5,
    # lr: 학습률 — 가중치를 한 번에 얼마나 크게 고칠지 (미리 학습된 모델을 살짝 고치는 거라 작게)
    lr=1e-4,
    # val_folds: 5등분 중 1조각을 검증용으로 → 약 80:20
    val_folds=5,
    # border_band: '테두리'로 볼 바깥 띠 두께 (한 변 길이의 10%)
    border_band=0.10,
)
S = CFG['img_size']
print(CFG)


# → 실험 설정을 한곳에 모아두겠다 (DACON E1b와 같은 것: ResNet18·ImageNet 가중치·전체 미세조정·224 입력)

In [ ]:
# ===== 5. File list & patient-level split =====


def list_images(folder):
    # 확장자가 이미지인 파일만 골라 정렬된 목록으로 돌려준다
    return sorted(p for p in glob.glob(f'{folder}/*') if p.lower().endswith(('.jpeg', '.jpg', '.png')))

def patient_id(path):
    # 파일 이름에서 환자 번호를 뽑는다
    name = os.path.basename(path)
    # 폐렴 사진 예: person1_bacteria_1.jpeg → 'person1'
    m = re.match(r'(person\d+)_', name)
    if m:
        return m.group(1)
    # 정상 사진 예: IM-0115-0001.jpeg → 'IM-0115', NORMAL2-IM-0123-0001.jpeg → 'NORMAL2-IM-0123'
    m = re.match(r'((?:NORMAL2-)?IM-\d+)-', name)
    if m:
        return m.group(1)
    # 규칙에 안 맞으면 파일 이름 자체를 한 명의 환자로 취급
    return name

def build_table(root):
    # 폴더 이름(NORMAL=0, PNEUMONIA=1)을 정답으로 하는 표를 만든다
    rows = []
    for label, cls in enumerate(['NORMAL', 'PNEUMONIA']):
        for p in list_images(f'{root}/{cls}'):
            rows.append(dict(path=p, label=label, patient=patient_id(p)))
    return pd.DataFrame(rows)

# train 폴더 → 학습+검증 후보, test 폴더 → 끝까지 손대지 않는 최종 시험용
df_all = build_table(TRAIN_DIR)
df_test = build_table(TEST_DIR)

# 5조각으로 나눈 것 중 첫 조각을 검증용으로 쓴다 (shuffle: 섞어서 나눔)
sgkf = StratifiedGroupKFold(n_splits=CFG['val_folds'], shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(df_all, df_all['label'], groups=df_all['patient']))
df_tr = df_all.iloc[tr_idx].reset_index(drop=True)
df_va = df_all.iloc[va_idx].reset_index(drop=True)

# 같은 환자가 train과 val에 동시에 들어갔는지 확인 (0이어야 정상)
print('overlap patients:', len(set(df_tr.patient) & set(df_va.patient)))
for name, d in [('train', df_tr), ('val', df_va), ('test', df_test)]:
    print(f'{name:5s} n={len(d):5d}  normal={int((d.label == 0).sum()):5d}  pneumonia={int(d.label.sum()):5d}')


# → 사진 목록과 정답표를 만들고, 같은 환자의 사진이 train·val에 나뉘지 않도록 환자 단위로 나누겠다
#   (test는 보통 normal 234 / pneumonia 390)

### 왜 환자 단위로 나누나?

Kermany 데이터는 한 환자가 사진을 여러 장 가진 경우가 많다 (`person1_bacteria_1`, `person1_bacteria_2` …).
사진 단위로 무작위로 나누면 **같은 환자의 사진이 train과 val에 동시에** 들어가고, 모델은 "이 사람 본 적 있다"로 맞혀서 검증 점수가 부풀려진다 (**데이터 누수, data leakage**).

| 나누는 방식 | 문제 |
|---|---|
| 사진 단위 무작위 | 같은 환자가 양쪽에 → 검증 점수가 실제보다 높게 나옴 |
| **환자 단위 (이 노트북)** | 검증 점수가 처음 보는 환자에 대한 성능에 가까움 |

In [ ]:
# ===== 6. EDA: image shape by class =====


def image_size(p):
    # Image.open은 그림 전체가 아니라 머리 부분만 읽어서 크기를 빨리 알려준다
    with Image.open(p) as im:
        return im.size

# 모든 사진의 가로·세로 크기를 모은다
paths_all = list(df_all.path) + list(df_test.path)
df_size = pd.DataFrame([image_size(p) for p in tqdm(paths_all, desc='size')], columns=['w', 'h'])
df_size['label'] = list(df_all.label) + list(df_test.label)
df_size['split'] = ['train'] * len(df_all) + ['test'] * len(df_test)
# aspect: 가로/세로 비율 (1보다 크면 가로로 긴 사진)
df_size['aspect'] = df_size.w / df_size.h
# pad_frac: 정사각형으로 만들 때 검은 여백이 차지하게 될 비율
df_size['pad_frac'] = 1 - np.minimum(df_size.w, df_size.h) / np.maximum(df_size.w, df_size.h)

# 정답(label)별 평균 — 0=NORMAL, 1=PNEUMONIA
df_size.groupby(['split', 'label'])[['w', 'h', 'aspect', 'pad_frac']].mean().round(3)


# → 정상/폐렴 사진의 모양(가로세로 비율)과, 정사각형으로 만들 때 생길 검은 여백의 양을 비교하겠다

### 이 표를 읽는 법

`pad_frac`(검은 여백 비율)이 **정상(0)과 폐렴(1)에서 다르면**, 여백의 두께 자체가 정답 힌트가 된다.
모델은 폐를 보지 않고 "여백이 두꺼우면 정상" 같은 **지름길(shortcut)** 을 배울 수 있다 — DACON에서 본 현상의 원인 후보.

| 결과 | 해석 |
|---|---|
| 두 클래스의 `pad_frac` 차이가 큼 | padded 모델이 테두리를 볼 동기가 강함 → 반전 장면이 잘 재현될 가능성 |
| 차이가 거의 없음 | 지름길이 약함 → 재현이 약하게 나올 수 있음 (그 결과도 그대로 보고) |

In [ ]:
# ===== 7. Preprocess: stretch vs letterbox =====


def load_gray(p):
    im = Image.open(p)
    # draft: JPEG를 원본 대신 작은 배율로 바로 해독 → 읽기 속도가 크게 빨라진다
    im.draft('L', (S * 2, S * 2))
    # 'L': 흑백 1채널로 통일 (일부 사진은 RGB로 저장되어 있음)
    return im.convert('L')

def stretch(im):
    # 비율을 무시하고 224×224로 늘리거나 줄인다 → 검은 여백 없음
    return im.resize((S, S), Image.BILINEAR)

def letterbox(im):
    # 비율을 지킨 채 긴 변을 224에 맞추고, 남는 곳을 검은색(0)으로 채운다 → DACON 학습 데이터와 같은 모양
    w, h = im.size
    r = S / max(w, h)
    small = im.resize((max(1, round(w * r)), max(1, round(h * r))), Image.BILINEAR)
    canvas = Image.new('L', (S, S), 0)
    # 가운데에 붙인다
    canvas.paste(small, ((S - small.width) // 2, (S - small.height) // 2))
    return canvas

def build_arrays(paths):
    # 각 사진을 두 방식으로 딱 한 번씩 변환해 메모리에 올려둔다 (에폭마다 다시 읽지 않아 빠르다)
    X_st = np.zeros((len(paths), S, S), np.uint8)
    X_lb = np.zeros((len(paths), S, S), np.uint8)
    for i, p in enumerate(tqdm(list(paths), desc='preprocess')):
        im = load_gray(p)
        X_st[i] = np.asarray(stretch(im))
        X_lb[i] = np.asarray(letterbox(im))
    return X_st, X_lb

# st = stretch(여백 없음), lb = letterbox(검은 여백)
Xtr_st, Xtr_lb = build_arrays(df_tr.path)
Xva_st, Xva_lb = build_arrays(df_va.path)
Xte_st, Xte_lb = build_arrays(df_test.path)
ytr, yva, yte = df_tr.label.values, df_va.label.values, df_test.label.values
print('train', Xtr_st.shape, ' val', Xva_st.shape, ' test', Xte_st.shape)


# → 모든 사진을 '늘린 버전'과 '검은 여백 버전' 두 가지 224×224로 미리 만들어 두겠다 (몇 분 걸림)

In [ ]:
# ===== 8. Dataset & DataLoader =====


# ImageNet 사진들의 평균·표준편차 — 미리 학습된 ResNet이 익숙한 밝기 범위로 맞춰준다
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class XrayDS(Dataset):
    def __init__(self, X, y, train=False):
        self.X, self.y, self.train = X, y, train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        # uint8(0~255) → float(0~1), 모양 [1, 224, 224]
        x = torch.from_numpy(self.X[i]).float().div(255).unsqueeze(0)
        if self.train:
            # 학습 때만 대비·밝기를 살짝 흔들어 외워버리기(과적합)를 줄인다
            # 회전·이동은 가장자리에 검은 여백을 새로 만들어 실험을 망치므로 일부러 뺐다
            x = (x * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05)).clamp(0, 1)
        # 흑백 1채널을 3번 복사해 ResNet이 기대하는 3채널로 만들고 정규화
        x = (x.repeat(3, 1, 1) - MEAN) / STD
        return x, int(self.y[i])

def make_loader(X, y, train):
    # shuffle: 학습 때만 순서를 섞는다, num_workers: 데이터를 준비하는 보조 프로세스 수
    return DataLoader(XrayDS(X, y, train), batch_size=CFG['batch_size'], shuffle=train,
                      num_workers=2, pin_memory=(DEVICE == 'cuda'))


# → 메모리의 사진 배열을 모델 입력 모양(3×224×224, 정규화)으로 바꿔 묶음 단위로 공급하는 도구를 만들겠다

In [ ]:
# ===== 9. Model & training loop =====


def build_model():
    # ImageNet으로 미리 학습된 ResNet18을 불러온다
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # 마지막 분류층(fc)을 1000개 클래스 → 2개(정상/폐렴)로 바꾼다
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, X):
    # 사진마다 '폐렴일 확률'(0~1)을 돌려준다, no_grad: 예측만 하니 기울기 계산을 끈다
    model.eval()
    probs = []
    for xb, _ in make_loader(X, np.zeros(len(X)), train=False):
        with torch.autocast(device_type='cuda', enabled=USE_AMP):
            out = model(xb.to(DEVICE))
        probs.append(out.float().softmax(1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

def train_model(name, Xtr, ytr, Xva, yva):
    model = build_model()
    # Adam: 많이 쓰는 최적화 방법, 모든 층의 가중치를 함께 고친다 (전체 미세조정 = E1b 방식)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
    # CrossEntropyLoss: 분류 문제의 기본 손실 함수
    loss_fn = nn.CrossEntropyLoss()
    # GradScaler: 16비트 계산에서 기울기가 너무 작아 사라지는 것을 막아준다
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best_auc, hist = -1, []
    for ep in range(1, CFG['epochs'] + 1):
        model.train(); t0 = time.time(); total = 0.0
        for xb, yb in tqdm(make_loader(Xtr, ytr, train=True), desc=f'{name} ep{ep}', leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            # autocast: 가능한 연산을 16비트로 → T4에서 학습이 빨라진다
            with torch.autocast(device_type='cuda', enabled=USE_AMP):
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += loss.item() * len(xb)
        # 에폭마다 검증 AUC를 재고, 가장 좋았던 가중치를 드라이브에 저장한다
        auc = roc_auc_score(yva, predict(model, Xva))
        hist.append(dict(model=name, epoch=ep, loss=total / len(Xtr), val_auc=auc, sec=time.time() - t0))
        print(f'{name} ep{ep}  loss={total / len(Xtr):.4f}  val_auc={auc:.4f}  ({time.time() - t0:.0f}s)')
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), f'{OUT_DIR}/{name}.pth')
    # 가장 좋았던 시점의 가중치로 되돌려 돌려준다
    model.load_state_dict(torch.load(f'{OUT_DIR}/{name}.pth', map_location=DEVICE))
    return model, hist


# → 모델을 만들고, 학습하면서 검증 AUC가 가장 좋았던 가중치를 저장하는 함수를 준비하겠다

In [ ]:
# ===== 10. Train two models =====


# 모델 A (clean): 학습·검증 모두 늘린 사진 → 검은 여백이 없다
model_clean, hist_clean = train_model('clean', Xtr_st, ytr, Xva_st, yva)
# 모델 B (padded): 학습·검증 모두 검은 여백 사진 → DACON 때의 학습 조건 재현
model_pad, hist_pad = train_model('padded', Xtr_lb, ytr, Xva_lb, yva)

df_hist = pd.DataFrame(hist_clean + hist_pad).round(4)
df_hist.to_csv(f'{OUT_DIR}/train_history.csv', index=False)
df_hist


# → 같은 ResNet18을 두 조건으로 각각 학습하겠다 (T4 기준 모델당 몇 분)

In [ ]:
# ===== 11. Test evaluation =====


def evaluate(name, model, X, y):
    p = predict(model, X)
    # 0.5 기준으로 폐렴(1) / 정상(0)을 나눈다
    pred = (p >= 0.5).astype(int)
    # ravel 순서: TN(정상→정상), FP(정상→폐렴), FN(폐렴→정상), TP(폐렴→폐렴)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(setting=name, acc=(tp + tn) / len(y), auc=roc_auc_score(y, p), FN=int(fn), FP=int(fp)), p

SETTINGS = [
    # 여백 없이 학습 → 여백 없는 시험
    ('clean / plain test', model_clean, Xte_st),
    # 여백 붙여 학습 → 여백 붙인 시험 (조건 일치)
    ('padded / padded test', model_pad, Xte_lb),
    # 여백 붙여 학습 → 여백 없는 시험 (DACON과 같은 상황)
    ('padded / plain test', model_pad, Xte_st),
]
rows = []
for name, model, X in SETTINGS:
    r, _ = evaluate(name, model, X, yte)
    rows.append(r)
df_eval = pd.DataFrame(rows).round(4)
df_eval.to_csv(f'{OUT_DIR}/test_eval.csv', index=False)
df_eval


# → 세 가지 조합으로 test 624장을 채점해 정확도·AUC·FN·FP를 비교하겠다

### 세 가지 시험 조합

| 조합 | 학습 사진 | 시험 사진 | 보고 싶은 것 |
|---|---|---|---|
| clean / plain test | 여백 없음 | 여백 없음 | 기준 성능 |
| padded / padded test | 검은 여백 | 검은 여백 | 학습·시험 조건이 같을 때 |
| **padded / plain test** | 검은 여백 | 여백 없음 | **DACON 상황** — 학습 때 의지하던 여백이 사라지면 무너지는가 |

- **FN** (폐렴인데 정상이라고 함): 의료에서 가장 위험한 실수
- **FP** (정상인데 폐렴이라고 함): DACON에서 테두리 집중이 높았던 그룹

In [ ]:
# ===== 12. Grad-CAM =====


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        # forward hook: 마지막 합성곱 묶음(layer4)을 지날 때 그 출력(512채널 × 7×7 특징 지도)을 잡아둔다
        self.handle = layer.register_forward_hook(self._save)

    def _save(self, module, inp, out):
        self.acts = out.detach()
        # 그 출력으로 거꾸로 흘러오는 기울기(gradient)도 잡아둔다
        out.register_hook(lambda g: setattr(self, 'grads', g.detach()))

    def __call__(self, X, cls=1):
        # cls=1: '폐렴' 점수를 기준으로 근거 지도를 만든다
        self.model.eval()
        cams, probs = [], []
        for xb, _ in make_loader(X, np.zeros(len(X)), train=False):
            xb = xb.to(DEVICE)
            self.model.zero_grad()
            out = self.model(xb)
            out[:, cls].sum().backward()
            # 채널별 기울기 평균 = 그 채널이 '폐렴' 판단에 얼마나 중요한지의 가중치
            w = self.grads.mean(dim=(2, 3), keepdim=True)
            # 가중치 × 특징 지도를 채널 방향으로 더하고, 음수는 0으로 (ReLU)
            cam = F.relu((w * self.acts).sum(1))
            # 사진마다 최댓값이 1이 되게 맞춘다
            cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
            cams.append(cam.float().cpu().numpy())
            probs.append(out.float().softmax(1)[:, 1].detach().cpu().numpy())
        return np.concatenate(cams), np.concatenate(probs)

    def remove(self):
        # 다 쓰면 hook을 떼어서 다른 계산에 영향이 없게 한다
        self.handle.remove()


# → 마지막 7×7 특징 지도와 그 기울기로, 모델이 사진의 어느 칸을 근거로 '폐렴'이라 했는지 계산하는 도구를 만들겠다

In [ ]:
# ===== 13. Border-focus ratio =====


def upsample(cams):
    # 7×7 지도를 224×224로 부드럽게(bilinear) 늘린다
    return F.interpolate(torch.from_numpy(cams)[:, None], size=S, mode='bilinear', align_corners=False)[:, 0].numpy()

def border_ratio(cams, band=CFG['border_band']):
    up = upsample(cams)
    # 바깥 띠(한 변의 10%)는 True, 안쪽은 False인 마스크
    b = int(S * band)
    mask = np.ones((S, S), bool)
    mask[b:-b, b:-b] = False
    # 전체 근거 중 바깥 띠가 차지하는 몫
    return up[:, mask].sum(1) / (up.reshape(len(up), -1).sum(1) + 1e-8)

# 기준선: 근거가 사진 전체에 고르게 퍼져 있으면 나오는 값 (= 바깥 띠의 넓이 비율)
print('uniform baseline:', round(1 - (1 - 2 * CFG['border_band']) ** 2, 3))

CAMS, rows = {}, []
for name, model, X in SETTINGS:
    gc = GradCAM(model, model.layer4)
    cams, probs = gc(X)
    gc.remove()
    CAMS[name] = (cams, probs, border_ratio(cams))
    pred = (probs >= 0.5).astype(int)
    for grp, sel in [('FP normal→pneumonia', (yte == 0) & (pred == 1)),
                     ('TN normal→normal', (yte == 0) & (pred == 0)),
                     ('TP pneumonia→pneumonia', (yte == 1) & (pred == 1))]:
        br = CAMS[name][2]
        rows.append(dict(setting=name, group=grp, n=int(sel.sum()),
                         border_ratio=float(br[sel].mean()) if sel.any() else np.nan))
df_border = pd.DataFrame(rows).round(3)
df_border.to_csv(f'{OUT_DIR}/border_ratio.csv', index=False)
df_border


# → 조합별·그룹별로 'Grad-CAM 근거 중 바깥 띠가 차지하는 비율'을 계산하겠다

### 테두리 집중 비율 읽는 법

- 근거가 사진 전체에 고르게 퍼져 있으면 값이 **기준선(uniform baseline, 띠 10%면 0.36)** 근처로 나온다.
- 기준선보다 **높으면** 모델이 가장자리를 유독 많이 봤다는 뜻, **낮으면** 안쪽(폐)에 집중했다는 뜻.
- DACON 보고서의 0.68 / 0.21은 띠 정의가 달랐을 수 있어 **숫자를 직접 비교하지 않는다.** 이 노트북 안에서 그룹끼리 비교한다.
- 반전이 약하게 나와도 그대로 기록한다 — 페이지에는 실제로 나온 숫자를 쓴다.

In [ ]:
# ===== 14. Export story.json for the web page =====


def thumb_b64(arr, n=180):
    # 224×224 사진을 180×180으로 줄여 PNG → base64 글자로 바꾼다 (웹 페이지 점 구름 = 180×180 격자)
    im = Image.fromarray(arr).resize((n, n), Image.BILINEAR)
    buf = io.BytesIO()
    im.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

def pick(sel, score):
    # 조건(sel)에 맞는 사진 중 score가 가장 큰 한 장의 번호
    idx = np.where(sel)[0]
    return int(idx[np.argmax(score[idx])]) if len(idx) else None

cams_c, probs_c, br_c = CAMS['clean / plain test']
cams_p, probs_p, br_p = CAMS['padded / plain test']

# 장면 4용: clean 모델이 폐렴을 맞힌 사진 중, 근거가 안쪽에 가장 몰린(테두리 비율이 가장 낮은) 한 장
i_good = pick((yte == 1) & (probs_c >= 0.5), -br_c)
# 장면 5용: padded 모델이 정상을 폐렴으로 틀린 사진 중 테두리 집중이 가장 큰 한 장 (틀린 게 없으면 정상 사진 전체에서)
sel_fp = (yte == 0) & (probs_p >= 0.5)
i_trick = pick(sel_fp if sel_fp.any() else (yte == 0), br_p)

def example(i, cams, probs, br):
    return dict(
        file=os.path.basename(df_test.path[i]),
        label=int(yte[i]),
        prob_pneumonia=round(float(probs[i]), 4),
        border_ratio=round(float(br[i]), 3),
        cam7=np.round(cams[i], 3).tolist(),
        image_plain=thumb_b64(Xte_st[i]),
        image_padded=thumb_b64(Xte_lb[i]),
    )

story = dict(
    source='Kermany, Zhang, Goldbaum (2018), Mendeley Data V2, doi:10.17632/rscbjbr9sj.2, CC BY 4.0',
    config=CFG,
    good=example(i_good, cams_c, probs_c, br_c),
    trick=example(i_trick, cams_p, probs_p, br_p),
    eval=df_eval.to_dict(orient='records'),
    border=df_border.to_dict(orient='records'),
)
with open(f'{OUT_DIR}/story.json', 'w') as f:
    # default: numpy 숫자처럼 json이 모르는 값은 파이썬 기본 숫자로 바꿔 저장
    json.dump(story, f, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('saved', f'{OUT_DIR}/story.json', round(os.path.getsize(f'{OUT_DIR}/story.json') / 1024), 'KB')
print('good :', story['good']['file'], story['good']['prob_pneumonia'], story['good']['border_ratio'])
print('trick:', story['trick']['file'], story['trick']['prob_pneumonia'], story['trick']['border_ratio'])


# → 웹 페이지에 들어갈 대표 사진 2장(잘 본 예 / 속은 예)의 사진·7×7 지도·점수와 표를 story.json 하나로 저장하겠다

In [ ]:
# ===== 15. Quick visual check =====


import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (title, i, cams) in zip(axes, [('clean model · pneumonia (good)', i_good, cams_c),
                                       ('padded model · normal (trick)', i_trick, cams_p)]):
    # 회색 X-ray 위에 붉은 근거 지도를 반투명하게 겹친다 (Reds: 단일 색상 계열)
    ax.imshow(Xte_st[i], cmap='gray')
    ax.imshow(upsample(cams[i][None])[0], cmap='Reds', alpha=0.45)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gradcam_examples.png', dpi=120)
plt.show()


# → 내보낸 두 장면을 X-ray + 붉은 Grad-CAM 겹친 그림으로 직접 눈으로 확인하겠다

## 다음 단계

1. 드라이브 `xray-gradcam-story/outputs/`에서 **`story.json`** 을 내려받는다
2. 로컬 `xray-gradcam-story/web/data/story.json` 에 넣는다
3. 웹 페이지가 합성 이미지 대신 이 파일의 사진·Grad-CAM을 읽도록 바꾼다

`outputs/` 안의 `.pth`(모델 가중치)는 용량이 커서 GitHub에 올리지 않는다 (`.gitignore`에 등록됨).
`test_eval.csv`, `border_ratio.csv`, `train_history.csv`, `gradcam_examples.png` 는 결과 기록용으로 `docs/`에 복사해 두면 좋다.